# 03.5 — Metadata and provenance

A chunk is a piece of text with no memory of where it came from. Everything it
knew about its own origin was thrown away the moment you sliced it.

Metadata is what you attach to put that back. This notebook is about deciding
*what* to attach — which is a question about your users, not about your
documents.

In [1]:
!pip install -q pymupdf4llm==1.28.2

## Metadata does exactly three jobs

Every field you attach is serving one of these. If it isn't, don't attach it.

**1. Citation.** The user asks a question, the system answers, and the user needs
to know where the answer came from. A chunk that can't name its source produces
an answer nobody can verify, and an unverifiable answer is one nobody can act on.

**2. Filtering.** Some questions are implicitly scoped — to a date range, a
product, a department, a language, a permission level. You can only filter on
what you captured.

**3. Deletion.** Someone exercises a right to erasure, or a document is withdrawn,
or a contract ends. You need to find every chunk derived from one document and
remove it. That requires a durable link back.

Citation and deletion need almost nothing — a stable identifier. Filtering is
where all the design work is, and it's the one people get wrong.

## The universal minimum

Four fields, and they apply whether you're indexing contracts, cookbooks or chat
logs.

In [2]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json

CORPUS = Path('../../corpus/docs')


def base_metadata(path, method, tool):
    return {
        'source': path.name,
        'content_hash': hashlib.sha256(path.read_bytes()).hexdigest()[:16],
        'ingested_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'extraction': method,           # text | ocr | vlm
        'extracted_by': tool,
    }

print(json.dumps(
    base_metadata(CORPUS / 'sahel-procurement-policy-v3.pdf', 'text', 'pymupdf4llm==1.28.2'),
    indent=2
))

{
  "source": "sahel-procurement-policy-v3.pdf",
  "content_hash": "03eb1249bd03a241",
  "ingested_at": "2026-09-12T14:45:28+00:00",
  "extraction": "text",
  "extracted_by": "pymupdf4llm==1.28.2"
}


`source` and `content_hash` cover citation and deletion.

The other three are operational, and they're the ones people skip. In module 02
you met a parser version that silently returned less than half of certain
documents. When you discover something like that, `extraction` and `extracted_by`
tell you exactly which chunks to rebuild. Without them you re-ingest everything
and hope.

`extraction` also carries a quality signal. A chunk marked `ocr` came from a
photocopy and should be trusted less than one marked `text` — and you may want to
surface that to the user rather than hide it.

That's the part that generalises. Everything below is about the third job.

## Deciding what to filter on

The instinct is to look at your documents and extract whatever they contain.
Document numbers, dates, authors, departments — it's all right there, so capture
it.

That's backwards. **Metadata is derived from the questions, not from the
documents.** A field nobody will ever filter on is storage, indexing cost and
maintenance for no return. A field somebody needs and you didn't capture means
re-ingesting the corpus.

So the procedure is: look at what users actually ask, work out what scoping each
question implies, and capture those fields.

### Worked from our question set

We have thirty-eight real questions. Read them and ask what filtering each one
would need.

In [3]:
import csv

questions = list(csv.DictReader(
    open('../../corpus/golden_questions.csv', newline='', encoding='utf-8')))

for q in questions[:6]:
    print(f"{q['id']}  {q['question']}")

Q01  How many working days of annual leave is a confirmed staff member entitled to?
Q02  What is the domestic per diem for staff travelling on Bank business?
Q03  How many days per week may Head Office staff work remotely?
Q04  How long is the probationary period?
Q05  How much annual leave may be carried into the following year and by when must it be used?
Q06  What notice period applies to confirmed staff on resignation?


Read those and the implied scoping falls out:

| Question shape | Implied filter |
| --- | --- |
| *How much leave do staff get?* | Which edition is current |
| *What does circular X change?* | Document identity |
| *Who approves a purchase of N?* | None — it's a plain lookup |
| *What was revenue in 2020?* | None — the year is in the text |

Only one shape actually needs a filter, and it needs the one thing these
documents don't state plainly: which version is in force.

That's a much shorter list than "everything the documents contain". No user asks
to be shown only documents produced by Qt 5.15, so there's no field for it.

### What this looks like elsewhere

Same procedure, different corpora, entirely different fields:

| Corpus | Questions people ask | Fields that implies |
| --- | --- | --- |
| Research papers | *What's the recent evidence on X?* | year, venue, authors |
| Product docs | *How do I do X in v3?* | product, version |
| Support tickets | *Has this bug been seen before?* | product, status, resolved date |
| Meeting transcripts | *What did Grace say about the budget?* | speaker, meeting, date |
| Textbooks | *Explain X* | chapter, section, page — for citation, not filtering |
| Customer emails | *What did we promise this client?* | account, sender, thread, date |

Note the textbook row. Its fields exist for citation only — nobody filters a
textbook by chapter, they just need to be told which chapter the answer came
from. Same three jobs, different job doing the work.

## The general problem underneath

One question shape in our set needed a filter, and it's worth naming what that
shape actually is, because it isn't specific to policies.

**Your corpus contains two documents that answer the same question differently,
and only one of them is right.**

That's the general problem. It turns up as:

- two editions of a policy, and the older one is void
- v2 and v3 of an API doc, and the user is on v3
- a spec and its errata
- a price list and a later discount schedule
- a draft and a signed contract
- a wiki page and the page that replaced it

In every case the retriever has no basis to choose, because both documents are
genuinely about the topic. It picks one. Sometimes it's the wrong one, and the
answer comes back fluent, correctly cited and wrong.

Many corpora don't have this at all. Two research papers disagreeing is the field
disagreeing, and you want both. Two support tickets describing different fixes is
history, not conflict. Before building anything here, check.

### How to check

Don't reason about it. Ask the corpus — which is what your evaluation set is for.

In [4]:
import pymupdf4llm, re

def find_conflicts(phrase, documents):
    """Which documents contain an answer to the same thing?"""
    hits = []
    for name in documents:
        text = pymupdf4llm.to_markdown(str(CORPUS / name))
        found = re.search(phrase, text)
        if found:
            hits.append((name, ' '.join(found.group(0).split())))
    return hits

for name, snippet in find_conflicts(
    r'entitled to[^.]{0,60}annual leave',
    [p.name for p in sorted(CORPUS.glob('*.pdf'))]
):
    print(f'{name}\n    {snippet}\n')

sahel-employee-handbook-2023.pdf
    entitled to **21 working days** of paid annual leave

sahel-employee-handbook-2025.pdf
    entitled to **25 working days** of paid annual leave



## Where the resolution comes from

You've established that two documents conflict. Now: which one wins?

The text will not tell you, reliably. It might in a particular corpus — ours does,
because a regulator writes in a house style — but that's a property of one
client's documents, not a technique you can carry anywhere.

In order of reliability:

**1. Ask the person putting the document in.** If there's an upload interface, a
single form field — *does this replace something?* — is worth more than any amount
of extraction. The uploader knows. Nobody else does.

**2. Ask the source system.** A document management system exists to track version
chains. SharePoint and Drive keep file history. Confluence has revisions. Git has
commits. A docs site has a version selector. Query it rather than inferring it.

**3. Use the file.** Folder structure, naming conventions, a `current/` directory.
Weak, but free, and often more truthful than the text.

**4. Read the documents.** Last resort. Works only when they happen to state it
consistently, which is a bet on one client's editorial standards.

Our corpus is a folder of files with no system and no uploader, so we're at level
4 — the worst case, not the normal one. In most engagements the answer already
exists in a system somebody has maintained for years, and the engineering job is
to fetch it rather than to reinvent it.

**Ask before you build.**

## The shape of the answer, wherever it comes from

However you obtain it, what you end up with is a small table that is *data*, not
code — because the people who know these answers are not engineers.

In [5]:
register = {
    # document                                  status         superseded by
    'sahel-employee-handbook-2023.pdf':      ('superseded', 'sahel-employee-handbook-2025.pdf'),
    'nfsc-circular-2024-07-cybersecurity.pdf': ('amended',   'nfsc-circular-2025-02-amendment.pdf'),
}

for doc, (status, by) in register.items():
    print(f'{status:<12} {doc}\n{"":<12} {doc}\n{"":<12} -> {by}\n')

superseded   sahel-employee-handbook-2023.pdf
             sahel-employee-handbook-2023.pdf
             -> sahel-employee-handbook-2025.pdf

amended      nfsc-circular-2024-07-cybersecurity.pdf
             nfsc-circular-2024-07-cybersecurity.pdf
             -> nfsc-circular-2025-02-amendment.pdf



Two entries. Not two thousand — because supersession is a property of document
*families*, and a corpus of ten thousand files is usually a few hundred families
with versions. You decide once per family.

In production this is a CSV the compliance team edits, or a table in the source
system, or a column in a DMS. Not a dictionary in a notebook. The one field it
must have that our example doesn't: **who confirmed it**, which is the difference
between a fact and a guess your pipeline is treating as a fact.

### Two statuses, not one

The distinction between those two entries is worth more than the mechanism.

**`superseded`** — every provision replaced. Don't retrieve it.

**`amended`** — parts changed, the rest still stands. Retrieve it, but whatever
uses it needs to know a later document exists.

The second one is where regulated industries actually live, and it's the one
people collapse into the first. Here's what collapsing it costs:

In [6]:
for doc in register:
    still_needed = [q for q in questions if q['source_document'] == doc]
    print(f'{doc}')
    print(f'    {len(still_needed)} questions still expect this document')
    for q in still_needed:
        print(f"      {q['id']}  {q['question'][:58]}")
    print()

sahel-employee-handbook-2023.pdf
    0 questions still expect this document

nfsc-circular-2024-07-cybersecurity.pdf
    2 questions still expect this document
      Q12  Can SMS one-time password alone satisfy multi-factor authe
      Q13  What penalty applies for missing a compliance date without



Zero questions need the old handbook — it's genuinely dead.

Two questions need the old circular, because the amendment changed two provisions
and left the rest in force.

Treat them the same way and you fix seven failures while silently breaking two
that currently work. **The overall score goes up.** You would not notice.

This is the argument for scoring per failure class rather than in aggregate, and
it's the most important thing in this notebook.

## Assembling it

Two layers. One you copy into every project, one you write fresh each time.

In [7]:
def corpus_metadata(name):
    """Specific to THIS corpus. Yours might contain different fields entirely."""
    status, superseded_by = register.get(name, ('current', None))
    return {'status': status, 'superseded_by': superseded_by}

name = 'nfsc-circular-2024-07-cybersecurity.pdf'
record = {
    **base_metadata(CORPUS / name, 'text', 'pymupdf4llm==1.28.2'),
    **corpus_metadata(name),
}
print(json.dumps(record, indent=2))

{
  "source": "nfsc-circular-2024-07-cybersecurity.pdf",
  "content_hash": "027a2a28c4141d91",
  "ingested_at": "2026-09-12T14:45:34+00:00",
  "extraction": "text",
  "extracted_by": "pymupdf4llm==1.28.2",
  "status": "amended",
  "superseded_by": "nfsc-circular-2025-02-amendment.pdf"
}


Five universal fields and two that exist because this corpus has a currency
problem. A corpus of research papers would have `year`, `venue` and `doi` in that
second slot and nothing about status.

Two more you'll want in production and we're not building today. An **owner or
department**, because access control has to filter on something — and if a
document is restricted to finance, that restriction has to reach your metadata or
your RAG system becomes a way to read files you can't open. And a **stable
document ID** rather than a filename, because deletion on request is a legal
obligation and filenames change.

## The rule that holds regardless of corpus

**Metadata is decided at ingestion and unrecoverable afterwards.**

If a field isn't captured when the document is parsed, adding it later means
re-ingesting and re-embedding everything. That's an afternoon at fifteen documents
and a project at ten thousand.

Which is why the question at the top of this notebook — *what will people ask, and
what scoping does that imply* — is worth an hour before you write any code.

## What's next

Notebook 6 assembles the whole module into a pipeline that runs unattended, and
then measures what any of it was worth.